In [ ]:
import pandas as pd
import requests
import json
import time
import concurrent.futures
from typing import List, Dict
import numpy as np

class OllamaAugmenter:
    def __init__(self, model="gemma3:12b", ollama_url="http://localhost:11434/api/generate"):
        self.model = model
        self.ollama_url = ollama_url

        # Optimal configuration based on Ollama behavior
        self.max_batch_size = 6  # Texts per batch
        self.max_concurrent_batches = 4  # Parallel batches
        self.max_total_tokens = 1500  # Per batch
    
    def create_smart_batches(self, df: pd.DataFrame) -> List[List[Dict]]:
        """Create optimized batches considering text lengths"""
        batches = []
        current_batch = []
        current_tokens = 0

        # Sort by text length for better batching
        df_sorted = df.copy()
        df_sorted['text_length'] = df_sorted['Text'].apply(len)
        df_sorted = df_sorted.sort_values('text_length')

        for _, row in df_sorted.iterrows():
            text = row['Text']
            # Approximate token count (Persian: ~1.3 chars per token)
            # 1 character ≈ 1.3 tokens
            estimated_tokens = len(text) / 1.3 + 100  # +100 for prompt overhead

            # Start new batch if:
            # 1. Too many tokens
            # 2. Too many texts
            # 3. Would exceed token limit
            if (len(current_batch) >= self.max_batch_size or current_tokens + estimated_tokens > self.max_total_tokens):
                if current_batch:
                    batches.append(current_batch)
                current_batch = []
                current_tokens = 0
            current_batch.append({
                'text': text,
                'score': row['Score'],
                'suggestion': row['Suggestion'],
                'original_idx': len(current_batch)
            })
            current_tokens += estimated_tokens
        if current_batch:
            batches.append(current_batch)

        print(f"Created {len(batches)} smart batches")
        return batches
    def create_batch_prompt(self, batch: List[Dict]) -> str:
        """Create optimized batch prompt"""

        prompt = """شما یک دستیار متخصص در تولید متن فارسی هستید. برای هر متن زیر، یک متن فارسی جدید با مفهوم مشابه ایجاد کن.

دستورالعمل‌ها:
1. هر متن جدید باید همان مفهوم و لحن متن اصلی را داشته باشد.
2. از کلمات و ساختار متفاوت استفاده کن.
3. طول متن مشابه باشد.
4. فقط متن جدید را بنویس.

ورودی:"""

        for i, item in enumerate(batch):
            label = {1: "مثبت", 3: "منفی", 2: "خنثی"}.get(item['suggestion'], "نامشخص")
            prompt += f"\n{i+1}. [{label}] {item['text']}"

        prompt += "\n\nخروجی (هر خط یک متن جدید):\n"
        return prompt
    def process_single_batch(self, batch: List[Dict], batch_id: int) -> List[Dict]:
        """Process a single batch"""
        prompt = self.create_batch_prompt(batch)

        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.7,
                "num_predict": 1000
            }
        }

        try:
            start_time = time.time()
            response = requests.post(self.ollama_url, json=payload, timeout=1200)

            if response.status_code == 200:
                result = response.json()
                response_text = result.get('response', '').strip()

                # Parse the response
                # Ollama response
                #       ↓
                # parse_batch_response()
                #       ↓
                # List[str]
                augmented_texts = self.parse_batch_response(response_text, len(batch))

                # Create results
                results = []
                for i, item in enumerate(batch):
                    augmented_text = augmented_texts[i] if i < len(augmented_texts) else item['text']
                    results.append({
                        'Text': augmented_text,
                        'Score': item['score'],
                        'Suggestion': item['suggestion'],
                        'Batch': batch_id,
                        'Time': time.time() - start_time
                    })

                print(f"✓ Batch {batch_id}: {len(batch)} texts in {time.time()-start_time:.1f}s")
                return results
            else:
                print(f"X Batch {batch_id}: Error {response.status_code}")
                return []

        except Exception as e:
            print(f"X Batch {batch_id}: {str(e)}")
            return []
    def parse_batch_response(self, response: str, expected_count: int) -> List[str]:
        """Parse batch response"""
        lines = [line.strip() for line in response.split('\n') if line.strip()]

        # Filter for Persian text
        persian_lines = []
        for line in lines:
            # Remove numbering
            if line.startswith(tuple(f"{i}." for i in range(1, 10))):
                line = line[2:].strip()

            # Check if it's Persian
            if any('\u0600' <= char <= '\u06FF' for char in line):
                persian_lines.append(line)
        return persian_lines[:expected_count]

    def augment_parallel_batches(self, df: pd.DataFrame) -> pd.DataFrame:
        """Main augmentation with parallel batch processing"""

        print(f"\nStarting parallel batch augmentation")
        print(f"Total texts: {len(df)}")
        print(f"Batch size: {self.max_batch_size}")
        print(f"Concurrent batches: {self.max_concurrent_batches}")
        print("-" * 60)

        # Create smart batches
        batches = self.create_smart_batches(df)

        # Process batches with ThreadPoolExecutor
        all_results = []
        batch_times = []

        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_concurrent_batches) as executor:
            # Submit all batches
            future_to_batch = {
                executor.submit(self.process_single_batch, batch, i): i
                for i, batch in enumerate(batches)
            }

            # Collect results as they complete
            for future in concurrent.futures.as_completed(future_to_batch):
                batch_id = future_to_batch[future]
                try:
                    batch_results = future.result(timeout=1500)
                    if batch_results:
                        all_results.extend(batch_results)
                        batch_times.extend([r['Time'] for r in batch_results])
                except Exception as e:
                    print(f"Batch {batch_id} failed: {e}")

            # Create DataFrame
            if all_results:
                augmented_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ['Batch', 'Time']} for r in all_results])
    
                # Statistics
                print(f"\n{'='*60}")
                print(f"COMPLETED: {len(augmented_df)} texts augmented")
                print(f"Total batches: {len(batches)}")
                print(f"Average time per batch: {np.mean(batch_times):.1f}s")
                print(f"Throughput: {len(augmented_df)/sum(batch_times):.2f} texts/sec")
    
                return augmented_df
            
            return pd.DataFrame()

augmenter = OllamaAugmenter(model="gemma3:12b")

df = pd.read_csv('digikala_sentiment.csv')

augmented_df = augmenter.augment_parallel_batches(df)